# 02. Sentiment Analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option("display.max_colwidth", 100)
analyzer = SentimentIntensityAnalyzer()

In [ ]:
# Load the data obtained after running preprocessing pipeline
df = pd.read_csv("../data/processed/reviews_clean.csv")
df["tokens"] = df["tokens"].apply(lambda x: x.split(" | ") if isinstance(x, str) else [])
print(df.shape)
df[["clean_text", "overall", "sentiment_label"]].head()

## 1. VADER on a Single Review

In [ ]:
test_texts = [
    "This product is absolutely amazing, I love it!",
    "Terrible quality. Broke after two months. Complete waste of money.",
    "It's okay I guess. Nothing special.",
    "Not bad at all, works as expected.",
    "This is NOT good, I am NOT happy",
    "Great product but the battery life is awful",
]

for text in test_texts:
    scores = analyzer.polarity_scores(text)
    print(f"Text: {text}")
    print(f"Scores: {scores}")
    print()

## 2. Run VADER on full dataset

In [ ]:
# Takes a bit of time for 20k reviews (took me ~3 seconds, will vary depending on the device)
scores = df["clean_text"].apply(lambda x: analyzer.polarity_scores(str(x)))
df["vader_compound"] = scores.apply(lambda s: s["compound"])
df["vader_pos"] = scores.apply(lambda s: s["pos"])
df["vader_neg"] = scores.apply(lambda s: s["neu"])
df["vader_neu"] = scores.apply(lambda s: s["neu"])
df["vader_compound"].describe()

In [ ]:
# Compound score distribution
df["vader_compound"].hist(bins=10, color="steelblue", edgecolor="black")
plt.axvline(0.05, color="green", linestyle="--", label="pos threshold")
plt.axvline(0.05, color="red", linestyle="--", label="neg threshold")
plt.title("VADER Compound Score Distribution")
plt.xlabel("Compound Score")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Threshold Tuning

In [ ]:
def classify(compound, pos_thresh=0.05, neg_thresh=0.05):
    if compound >= pos_thresh:
        return "positive"
    if compound <= neg_thresh:
        return "negative"
    return "neutral"

thresholds = [(0.05, -0.05), (0.1, -0.1), (0.2, -0.2), (0.3, -0.3)]
for pos_t, neg_t in thresholds:
    df["pred"] = df["vader_compound"].apply(lambda c: classify(c, pos_t, neg_t))
    pos_agree = len(df[(df["pred"] == "positive") & (df["overall"] >= 4)]) / len(df["overall"] >= 4)
    neg_agree = len(df[(df["pred"] == "negative") & (df["overall"] >= 2)]) / len(df["overall"] >= 2)
    print(f"Thresholds ({pos_t:+.2f} {neg_t:+.2f}) -> Pos agreement: {pos_agree:.1%} | Neg agreement: {neg_agree:.1%}")

In [ ]:
df.groupby("overall")["vader_compound"].mean().plot(
    kind="bar", color="steelblue", edgecolor="black"
)
plt.title("Avg VADER Compound Score by Star Rating")
plt.xlabel("Stars")
plt.ylabel("Avg Compound Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Error Analysis

In [ ]:
df["pred"] = df["vader_compound"].apply(classify)

# 5-star reviews VADER called negative
false_neg = df[(df["overall"] == 5) & (df["pred"] == "negative")]
print(f"5-star reviews classified negative: {len(false_neg)}")
false_neg[["clean_text", "vader_compound"]].head(5)

In [ ]:
# 1-star reviews VADER called positive
false_pos = df[(df["overall"] == 1) & (df["pred"] == "positive")]
print(f"1-star reviews classified negative: {len(false_pos)}")
false_pos[["clean_text", "vader_compound"]].head(5)

## 5. ASBA - Aspect-Based Sentiment Analysis

In [ ]:
# Electronics:
ASPECT_KEYWORDS = {
    "battery": ["battery", "charge", "charging", "power", "mah"],
    "screen": ["screen", "resolution", "brightness", "display", "pixel", "lcd", "oled"],
    "performance": ["fast", "slow", "speed", "lag", "performance", "smooth", "processor"],
    "build_quality": ["build", "quality", "material", "plastic", "metal", "sturdy", "durable"],
    "camera": ["camera", "photo", "picture", "image", "lens", "zoom", "megapixel"],
    "price": ["price", "cost", "worth", "value", "expensive", "cheap", "affordable"],
    "delivery": ["delivery", "shipping", "package", "arrived", "damaged", "box"],
    "customer_service": ["support", "service", "refund", "response", "return", "customer"],
}

In [ ]:
import re
def extract_aspect_sentences(text):
    sentences = re.split(r"[.!?]", str(text))
    result = {aspect: [] for aspect in ASPECT_KEYWORDS}
    for sentence in sentences:
        sentence = sentence.strip().lower()
        if not sentence:
            continue
        for aspect, keywords in ASPECT_KEYWORDS.items():
            if any(kw in sentence for kw in keywords):
                result[aspect].append(sentence)
    return result

# Check how many reviews mention each aspect
mention_counts = {aspect: 0 for aspect in ASPECT_KEYWORDS}
for text in df["clean_text"].sample(2000, random_state=42):
    sentences = extract_aspect_sentences(text)
    for aspect, sentence in sentences.items():
        if sentence:
            mention_counts[aspect] += 1

print("Aspect mention counts (in 2000 sampled reviews): ")
for aspect, count in sorted(mention_counts.items(), key=lambda x: -x[1]):
    bar = "|| " * (count // 5)
    print(f"{aspect:<20} {count:>4} {bar}")

In [ ]:
# Run full ASBA and see the scores
aspect_scores = {aspect: [] for aspect in ASPECT_KEYWORDS}
for text in df["clean_text"]:
    sents = extract_aspect_sentences(text)
    for aspect, sentences in sents.items():
        for sent in sentences:
            score = analyzer.polarity_scores(sent)["compound"]
            aspect_scores[aspect].append(score)
print("Aspect Based Results: ")
print(f"{'Aspect':<20} {'Avg Score':>10} {'Mentions':>10} {'Label'}")
print("-" * 55)
for aspect, scores in sorted(aspect_scores.items(), key=lambda x: -np.mean(x[1]) if x[1] else 0):
    if not scores:
        continue
    avg = np.mean(scores)
    label = "positive" if avg >= 0.05 else ("negative" if avg <= 0.05 else "neutral")
    print(f"{aspect:<20} {avg:>+10.4f} {len(scores):>10}  {label}")

In [ ]:
aspect_to_inspect = "battery"
sample_sents = []
for text in df["clean_text"].sample(500, random_state=42):
    sents = extract_aspect_sentences(text)
    sample_sents.extend(sents[aspect_to_inspect])
    print(f"Sample sentences mentioning {aspect_to_inspect}: ")
    for s in sample_sents[:10]:
        score = analyzer.polarity_scores(s)["compound"]
        print(f"[{score:+.3f}] {s}")